# Aggregation Function Distance Metric — L5 Analysis

For each L5 case we:
1. Find source CSVs with many rows and a target with few rows (aggregation happened)
2. Find numerical columns shared between source and target
3. For each shared column, build a **5×5 per-column distance matrix** over `{min, max, mean, sum, count}`
4. Average all per-column matrices across 50 cases → one final matrix per metric

Entry `[i, j]` = expected distance if the correct aggregation is `i` but `j` was predicted,
estimated from 50 real L5 cases.

Metrics: **L1, L2, KL Divergence, EMD, JS Distance (log base 2)**

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
import warnings
warnings.filterwarnings('ignore')

BASE = '/home/asurite.ad.asu.edu/jrtandel/transchema/autopipeline-benchmarks/github-pipelines/'
AGG_FUNCS  = {'min': 'min', 'max': 'max', 'mean': 'mean', 'sum': 'sum', 'count': 'count'}
AGG_NAMES  = list(AGG_FUNCS.keys())
N_CASES    = 50

In [ ]:
# --------------------------------------------------------------------------
# Discover L5 cases: source has many rows, target has few rows,
# and they share at least one numerical column (exact name match)
# --------------------------------------------------------------------------

case_specs = []  # list of {case, src_path, tgt_path, shared_num_cols}

for d in sorted(os.listdir(BASE)):
    if not d.startswith('length5_'):
        continue
    path = os.path.join(BASE, d)
    tgt_path = os.path.join(path, 'target.csv')
    if not os.path.exists(tgt_path):
        continue
    try:
        target = pd.read_csv(tgt_path, index_col=0, low_memory=False)
    except:
        continue
    if len(target) >= 30:
        continue

    tgt_num = set(target.select_dtypes(include='number').columns.str.lower())
    if not tgt_num:
        continue

    for src_file in sorted(os.listdir(path)):
        if not src_file.startswith('test_') or not src_file.endswith('.csv'):
            continue
        src_path = os.path.join(path, src_file)
        try:
            src = pd.read_csv(src_path, index_col=0, low_memory=False)
        except:
            continue
        if len(src) < 50:
            continue

        src_num  = set(src.select_dtypes(include='number').columns.str.lower())
        shared   = sorted(tgt_num & src_num)
        if shared:
            case_specs.append({
                'case':     d.replace('length', ''),
                'src_path': src_path,
                'tgt_path': tgt_path,
                'shared_cols': shared
            })

case_specs = case_specs[:N_CASES]
print(f'Using {len(case_specs)} case+source combinations')
for cs in case_specs:
    print(f"  {cs['case']:6s} | {os.path.basename(cs['src_path'])} | cols: {cs['shared_cols']}")

In [ ]:

# --------------------------------------------------------------------------
# Distance functions — all return [0, 1]
# Histogram bin range = only the two arrays being compared (per-pair)
# Bin count = Sturges' rule: ceil(log2(n) + 1)
# --------------------------------------------------------------------------

def to_prob_shared(a, b):
    n    = len(a)
    bins = int(np.ceil(np.log2(n) + 1))
    lo   = min(a.min(), b.min())
    hi   = max(a.max(), b.max())
    if lo == hi:
        return np.array([1.0]), np.array([1.0])
    pa, _ = np.histogram(a, bins=bins, range=(lo, hi))
    pb, _ = np.histogram(b, bins=bins, range=(lo, hi))
    pa = pa.astype(float) + 1e-10
    pb = pb.astype(float) + 1e-10
    return pa / pa.sum(), pb / pb.sum()

def value_range(a, b):
    lo = min(a.min(), b.min())
    hi = max(a.max(), b.max())
    return (hi - lo) if (hi - lo) > 0 else 1.0

def dist_l1(a, b):
    return np.mean(np.abs(a - b)) / value_range(a, b)

def dist_l2(a, b):
    return np.sqrt(np.mean((a - b) ** 2)) / value_range(a, b)

def dist_emd(a, b):
    from scipy.stats import wasserstein_distance
    return wasserstein_distance(a, b) / value_range(a, b)

def dist_kl(a, b):
    from scipy.stats import entropy
    pa, pb = to_prob_shared(a, b)
    return 1 - np.exp(-entropy(pa, pb))

def dist_js(a, b):
    from scipy.spatial.distance import jensenshannon
    pa, pb = to_prob_shared(a, b)
    return float(jensenshannon(pa, pb, base=2))

def dist_value_ratio(a, b):
    """Range overlap similarity: fraction of [min_a,max_a] that overlaps [min_b,max_b].
    Returns 1.0 when ranges are identical, 0.0 when they don't overlap at all.
    Expressed as a distance: 1 - overlap so 0 = identical, 1 = no overlap."""
    gen_min, gen_max = float(a.min()), float(a.max())
    gt_min,  gt_max  = float(b.min()), float(b.max())
    overlap = max(0.0, min(gen_max, gt_max) - max(gen_min, gt_min))
    union   = max(gen_max, gt_max) - min(gen_min, gt_min)
    range_overlap = round(overlap / union, 4) if union > 0 else 1.0
    return 1.0 - range_overlap  # convert similarity → distance

METRICS = {
    'L1':            dist_l1,
    'L2':            dist_l2,
    'KL Divergence': dist_kl,
    'EMD':           dist_emd,
    'JS Distance':   dist_js,
    'Value Ratio':   dist_value_ratio,
}
print('Metrics:', list(METRICS.keys()))


In [ ]:
# --------------------------------------------------------------------------
# For each (case, col), apply each aggregation to the full column → scalar
# Store RAW (unnormalized) scalars — normalization happens per (agg_i, agg_j) pair
# --------------------------------------------------------------------------

agg_scalars_raw = {name: [] for name in AGG_NAMES}  # raw scalar per instance
instance_pairs_raw = []  # dict of agg→raw_scalar per (case, col)

for cs in case_specs:
    try:
        src = pd.read_csv(cs['src_path'], index_col=0, low_memory=False)
    except:
        continue

    for col in cs['shared_cols']:
        col_actual = next((c for c in src.columns if c.lower() == col), None)
        if col_actual is None:
            continue

        vals = pd.to_numeric(src[col_actual], errors='coerce').dropna().values
        if len(vals) == 0:
            continue

        instance = {}
        for name, func in AGG_FUNCS.items():
            if func == 'count':
                scalar = float(len(vals))
            else:
                scalar = float(getattr(pd.Series(vals), func)())
            instance[name] = scalar
            agg_scalars_raw[name].append(scalar)

        instance_pairs_raw.append(instance)

print(f'Collected {len(instance_pairs_raw)} (case, col) instances (raw, unnormalized)')
print('Raw scalar ranges per aggregation:')
for name in AGG_NAMES:
    arr = np.array(agg_scalars_raw[name])
    print(f'  {name:6s}: [{arr.min():.2f}, {arr.max():.2f}]  mean={arr.mean():.2f}')

In [ ]:
# --------------------------------------------------------------------------
# Build the 5×5 matrices — each (agg_i, agg_j) pair normalizes independently
#
# For each entry [i, j]:
#   A = raw scalars of agg_i across all instances
#   B = raw scalars of agg_j across all instances
#   pair_range = max(A ∪ B) - min(A ∪ B)   ← local to this pair only
#
#   L1 / L2 / EMD : mean(|A - B|) / pair_range
#   KL / JS       : histogram of A_norm vs B_norm, where norm = (x - pair_lo) / pair_range
# --------------------------------------------------------------------------

def build_matrix_per_pair(instance_pairs_raw, agg_scalars_raw, metric_name, metric_fn):
    n   = len(AGG_NAMES)
    mat = np.zeros((n, n))

    for i, name_i in enumerate(AGG_NAMES):
        for j, name_j in enumerate(AGG_NAMES):
            A = np.array(agg_scalars_raw[name_i])
            B = np.array(agg_scalars_raw[name_j])

            # per-pair normalization
            pair_lo    = min(A.min(), B.min())
            pair_hi    = max(A.max(), B.max())
            pair_range = (pair_hi - pair_lo) if pair_hi > pair_lo else 1.0

            A_norm = (A - pair_lo) / pair_range
            B_norm = (B - pair_lo) / pair_range

            if metric_name in ('KL Divergence', 'JS Distance'):
                mat[i, j] = metric_fn(A_norm, B_norm)
            else:
                # L1 / L2 / EMD on normalized values
                mat[i, j] = metric_fn(A_norm, B_norm)

    return pd.DataFrame(mat, index=AGG_NAMES, columns=AGG_NAMES)

avg_matrices = {}
for metric_name, metric_fn in METRICS.items():
    avg_matrices[metric_name] = build_matrix_per_pair(
        instance_pairs_raw, agg_scalars_raw, metric_name, metric_fn
    )

print('Matrices built — each (agg_i, agg_j) pair uses its own normalization range.')
print('\nExample — L1 matrix:')
avg_matrices['L1'].round(4)

In [ ]:
# Sort axes by mean raw scalar value across all instances
agg_global_mean  = {name: np.mean(agg_scalars_raw[name]) for name in AGG_NAMES}
AGG_NAMES_SORTED = sorted(agg_global_mean, key=agg_global_mean.get)

print('Aggregation functions sorted by mean raw output (across all L5 instances):')
for name in AGG_NAMES_SORTED:
    print(f'  {name:6s}: {agg_global_mean[name]:.2f}')

In [ ]:

# --------------------------------------------------------------------------
# Visualize averaged matrices — axes sorted by mean output value
# --------------------------------------------------------------------------

n_metrics = len(avg_matrices)
fig, axes = plt.subplots(1, n_metrics, figsize=(n_metrics * 5.5, 5))

for idx, (metric_name, mat) in enumerate(avg_matrices.items()):
    ax    = axes[idx]
    mat_s = mat.loc[AGG_NAMES_SORTED, AGG_NAMES_SORTED]

    sns.heatmap(
        mat_s, annot=True, fmt='.3f', cmap='YlOrRd',
        ax=ax, linewidths=0.5, square=True,
        cbar_kws={'shrink': 0.8},
        vmin=0, vmax=1
    )
    ax.set_title(metric_name, fontsize=13, fontweight='bold')
    ax.set_xlabel('Predicted Aggregation', fontsize=9)
    ax.set_ylabel('Correct Aggregation',   fontsize=9)
    ax.tick_params(axis='both', labelsize=9)

plt.suptitle(
    f'Average Aggregation Distance Matrices — {N_CASES} L5 cases\n'
    'Entry[i,j] = avg distance when correct agg=row but predicted agg=col\n'
    'Axes sorted by mean output value (low → high)',
    fontsize=12, y=1.03
)
plt.tight_layout()
plt.savefig('agg_distance_l5_avg.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: agg_distance_l5_avg.png')


In [ ]:
# --------------------------------------------------------------------------
# Print all averaged matrices
# --------------------------------------------------------------------------

for metric_name, mat in avg_matrices.items():
    print(f'\n===== {metric_name} =====')
    print(mat.loc[AGG_NAMES_SORTED, AGG_NAMES_SORTED].round(4).to_string())

In [ ]:
# --------------------------------------------------------------------------
# Ranking: for each correct agg, which predicted agg is least / most wrong?
# --------------------------------------------------------------------------

print('For each metric, given correct agg = ROW, which predicted agg is LEAST wrong?\n')

for metric_name, mat in avg_matrices.items():
    mat_s = mat.loc[AGG_NAMES_SORTED, AGG_NAMES_SORTED]
    print(f'--- {metric_name} ---')
    for row in AGG_NAMES_SORTED:
        others  = mat_s.loc[row].drop(row).dropna()
        if others.empty:
            continue
        closest  = others.idxmin()
        farthest = others.idxmax()
        print(f'  correct={row:5s} | closest wrong={closest:5s} ({others[closest]:.4f}) | '
              f'farthest wrong={farthest:5s} ({others[farthest]:.4f})')
    print()